<a href="https://colab.research.google.com/github/abhijadhav14/Data-Analytics-Using-Python/blob/main/K_Means_Clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# K-Means Clustering using PySpark MLlib

# Import Libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import randn
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Initialize Spark Session
spark = SparkSession.builder.appName("KMeansLargeData").getOrCreate()

# 1. Dataset (100,000 rows, 3 features)


df = spark.range(0, 100000).select(
    (randn(seed=42) * 10).alias("feature1"),
    (randn(seed=43) * 50).alias("feature2"),
    (randn(seed=44) * 100).alias("feature3")
)

# 2. Combine Features into a Single Vector
assembler = VectorAssembler(
    inputCols=["feature1", "feature2", "feature3"],
    outputCol="raw_features"
)
df_assembled = assembler.transform(df)

# 3. Scale the Features (Crucial for distance-based algorithms like K-Means)
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,
    withMean=True
)
scaler_model = scaler.fit(df_assembled)
final_df = scaler_model.transform(df_assembled)

# 4. Create and Train K-Means Model

kmeans = KMeans(featuresCol="features", predictionCol="cluster", k=4, seed=42)
kmeans_model = kmeans.fit(final_df)

# 5. Make Predictions (Assign data points to clusters)
predictions = kmeans_model.transform(final_df)

# 6. Evaluate Model (Using Silhouette Score)
evaluator = ClusteringEvaluator(
    predictionCol="cluster",
    featuresCol="features",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)
silhouette_score = evaluator.evaluate(predictions)
print(f"Silhouette Score: {silhouette_score}")

# 7. Show Cluster Centers
print("\nCluster Centers:")
centers = kmeans_model.clusterCenters()
for center in centers:
    print(center)

# 8. View Example Predictions
print("\nSample Cluster Assignments:")
predictions.select("feature1", "feature2", "feature3", "cluster").show(5)

Generating large dataset...
Silhouette Score: 0.3535354876838639

Cluster Centers:
[-0.57793417 -1.03529418 -0.15355303]
[-0.63905543  0.69459635  0.72766458]
[ 1.05733094 -0.21001533  0.48754849]
[ 0.14176065  0.53294572 -1.04342807]

Sample Cluster Assignments:
+------------------+-------------------+------------------+-------+
|          feature1|           feature2|          feature3|cluster|
+------------------+-------------------+------------------+-------+
| 23.84479054241165|  55.13527240727682|29.214347275992463|      2|
| 1.920934041293524| 28.605223118377275| 36.00856476365026|      1|
|7.3373365332865745|-33.268062633206405|3.3737520764796156|      2|
|-5.224480195716871| -39.18902061184852| 145.3407919198987|      0|
| 20.60084179317831|  19.60167589073964| 92.40505873847208|      2|
+------------------+-------------------+------------------+-------+
only showing top 5 rows
